# EDA Principal — Fase 4
## Análisis Exploratorio de Datos: Regulación IA y Ecosistemas de IA en 199 Países

---

**Proyecto:** Research AI Law — IMT3860 (Taller de Investigación, PUC Chile, abril 2026)  
**Caso focal:** Boletín 16821-19 — Proyecto de Ley Marco de Inteligencia Artificial de Chile (Senado)  
**Fecha:** 2026-05-06  
**Autor:** Pablo Sturias / tiasturias  
**Versión datos:** Matriz Madre Fase 3 v1.1 — 199 países × 1,203 columnas

---

## Pregunta de Investigación Principal

> *¿Existe una asociación estadísticamente significativa entre las características de la regulación de IA de un país y el desarrollo de su ecosistema de IA, después de controlar por factores socioeconómicos e institucionales?*

---

## Alcance y límites de este EDA

Este notebook es el **EDA Principal de Fase 4**. Su propósito es **explorar, describir y preparar** — no responder causalmente la pregunta principal. Concretamente:

- **Hace:** Diagrama la calidad de los datos, identifica variables con suficiente cobertura, detecta correlaciones bivariadas, perfila a Chile en el contexto global.
- **No hace:** Imputar datos, estimar modelos causales, decidir la muestra final, ni interpretar correlaciones como causalidad.

Los modelos econométricos son Fase 6. La selección final de muestra y variables es Fase 5+6.

---

## Estructura del Notebook

| Bloque | Contenido |
|--------|----------|
| **Setup** | Paths, imports, configuración |
| **A** | Contrato de datos — verificar que la Matriz Madre es correcta |
| **B** | Cobertura y missingness por país, variable, bloque, región |
| **C** | Estadística univariada robusta — distribuciones, asimetría, outliers |
| **D** | Análisis intra-bloque — variables más informativas por bloque |
| **E/F** | Correlaciones Spearman + corrección FDR, redundancia |
| **I** | Perfiles de país — Chile, LATAM, SGP/ARE/IRL |
| **L** | Submuestras candidatas — multiverse analysis preregistrado |
| **M** | Recomendaciones para Fase 5 — gaps, transformaciones, decisiones |

In [ ]:
# =============================================================================
# SETUP: Paths
# Resuelve FASE4/ y FASE3/ y expone sus paquetes públicos sin leer CSVs directos.
# =============================================================================
import sys, os
from pathlib import Path

cwd = Path().resolve()
if cwd.name == 'notebooks':
    FASE4_ROOT = cwd.parent
elif (cwd / 'src' / 'fase4').exists():
    FASE4_ROOT = cwd
else:
    FASE4_ROOT = Path('/home/pablo/Research_LeyIA_DataScience/Research_AI_law/FASE4')
FASE3_ROOT = FASE4_ROOT.parent / 'FASE3'

# Importar fase4 como paquete directo evita colisiones con el paquete src de Fase 3.
for path in [FASE4_ROOT / 'src', FASE4_ROOT, FASE3_ROOT, FASE3_ROOT / 'src']:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

os.chdir(str(FASE4_ROOT))

print(f"FASE4_ROOT: {FASE4_ROOT}")
print(f"FASE3_ROOT: {FASE3_ROOT}")
print(f"cwd:        {os.getcwd()}")
assert FASE4_ROOT.exists(), f"FASE4_ROOT no existe: {FASE4_ROOT}"
assert FASE3_ROOT.exists(), f"FASE3_ROOT no existe: {FASE3_ROOT}"
print("Paths configurados correctamente")


In [ ]:
# =============================================================================
# SETUP: Imports
# Bibliotecas estándar + módulos src/fase4/. El notebook orquesta; la lógica vive en src/fase4/.
# =============================================================================
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')  # backend robusto para ejecución reproducible con nbconvert
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
pd.set_option('display.float_format', lambda x: f'{x:.3f}')
matplotlib.rcParams['figure.figsize'] = (12, 6)
matplotlib.rcParams['font.size'] = 11
sns.set_theme(style='whitegrid')

from fase4.contracts import run_contracts
from fase4.load import (
    load_wide, load_dictionary, current_version,
    get_variable_cols, get_block_var_cols
)
from fase4.coverage import run_coverage_analysis
from fase4.univariate import run_univariate_analysis
from fase4.blocks import run_block_analysis
from fase4.bivariate import run_bivariate_analysis
from fase4.question_mapping import run_question_mapping
from fase4.binding import run_taxonomy_analysis
from fase4.countries import run_country_analysis
from fase4.temporal import run_temporal_analysis
from fase4.submuestras import run_submuestras_analysis
from fase4.reporting import run_reporting
from fase4.config import BLOCKS, OUTPUTS_DIR

print("Imports exitosos")



---
## Bloque A — Contrato de Datos

Antes de cualquier análisis, verificamos que los datos cargados son los que esperamos.
Esto se llama **contrato de datos**: un conjunto de afirmaciones explícitas que deben
cumplirse para que el análisis sea válido.

Si alguna de estas afirmaciones falla, **el notebook debe detenerse** — significa que
algo cambió en la Matriz Madre y los análisis anteriores podrían estar desactualizados.

**Afirmaciones que verificamos:**
1. La Matriz Madre tiene exactamente 199 países (filas).
2. La Matriz Madre tiene 1,203 columnas.
3. Chile (iso3=`CHL`) está presente en los datos.
4. Los 6 bloques temáticos están representados en el diccionario.

In [ ]:
# Cargar la Matriz Madre y el diccionario de variables
wide = load_wide()
dictionary = load_dictionary()

print(f"Versión Fase 3: {current_version()}")
print(f"Dimensiones wide: {wide.shape[0]} países × {wide.shape[1]} columnas")
print(f"Variables en diccionario: {len(dictionary)}")
print()

# ── Contrato de datos ────────────────────────────────────────────────────────
assert wide.shape[0] == 199, f"FALLA: se esperan 199 países, hay {wide.shape[0]}"
assert wide.shape[1] == 1203, f"FALLA: se esperan 1203 columnas, hay {wide.shape[1]}"
assert 'CHL' in wide['iso3'].values, "FALLA: Chile (CHL) no está en los datos"
assert 'iso3' in wide.columns, "FALLA: columna iso3 no encontrada"

# Verificar que los 6 bloques están en el diccionario
bloques_en_dict = set(dictionary['bloque_tematico'].dropna().unique())
for bloque in BLOCKS:
    assert bloque in bloques_en_dict, f"FALLA: bloque '{bloque}' no está en el diccionario"

print("✓ Contrato de datos: TODOS los checks pasaron")
print()
print(f"Bloques temáticos en diccionario:")
for b in BLOCKS:
    n_vars = len(dictionary[dictionary['bloque_tematico'] == b])
    print(f"  {b}: {n_vars} variables")

# Guardar contrato formal y hashes de inputs Fase 3 para trazabilidad.
contracts_results = run_contracts(wide=wide, dictionary=dictionary, save=False)
display(contracts_results['contract_check'])
print("Contrato formal calculado en memoria; los outputs oficiales no se sobrescriben.")



In [ ]:
# Vista previa: columnas de identidad de los primeros 10 países
id_cols = ['iso3', 'country_name_canonical', 'region', 'income_group',
           'n_sources_present', 'pct_variables_available', 'included_in_dense_candidate']
id_cols_present = [c for c in id_cols if c in wide.columns]

print("Vista previa — columnas de identidad (primeras 10 filas):")
display(wide[id_cols_present].head(10))

In [ ]:
# Verificar que Chile está en los datos y mostrar sus metadatos
chile_row = wide[wide['iso3'] == 'CHL'][id_cols_present]
print("Registro de Chile en la Matriz Madre:")
display(chile_row)

---
## Bloque B — Cobertura y Missingness

### ¿Por qué importa el missingness?

En ciencias sociales cuantitativas, los datos faltantes no son un problema técnico menor:
son **información sobre qué países tienen capacidad de medir y reportar**. Un país sin datos
en el bloque `regulatory_treatment` puede significar dos cosas muy distintas:

1. **No tiene regulación de IA** (y por eso no reporta a IAPP).
2. **Tiene regulación pero no está rastreada** (porque es menor o en idioma no indexado).

Ambas interpretaciones tienen consecuencias diferentes para el análisis causal.

### Cuello de botella estructural

El bloque `regulatory_treatment` (datos IAPP — International Association of Privacy Professionals)
tiene datos para **~28 de 199 países (~14%)**.  Esto es el **cuello de botella estructural**
de toda la investigación: la variable de tratamiento principal (¿tiene el país regulación de IA?)
solo existe para una fracción pequeña de los países.

Las implicaciones son:
- El modelo causal principal operará sobre una muestra de ~28 países (submuestra `regulada`).
- Se necesitan análisis de sensibilidad con muestras más grandes pero con tratamiento más débil.
- Los rankings de Chile deben interpretarse en el contexto de qué bloque se mide.

In [ ]:
# Ejecutar análisis completo de cobertura
# Esta función calcula missingness por variable, país, bloque y región
print("Ejecutando análisis de cobertura...")
coverage_results = run_coverage_analysis(wide=wide, dictionary=dictionary, save=False)
print("✓ Análisis de cobertura completado")
print(f"  Tablas generadas: {list(coverage_results.keys())}")


In [ ]:
# Resumen global de calidad de la Matriz Madre
quality_df = coverage_results['eda_quality_overview']
print("Resumen global de calidad — Matriz Madre Fase 3 v1.1:")
display(quality_df)


In [ ]:
# Cobertura por bloque temático — tabla detallada
miss_block = coverage_results['eda_missingness_by_block']
print("Cobertura por bloque temático:")
display(miss_block)


In [ ]:
# Gráfico de barras: países con datos por bloque
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Panel izquierdo: N países con datos
ax = axes[0]
colors = ['#c0392b' if v < 50 else '#e67e22' if v < 100 else '#27ae60'
          for v in miss_block['n_countries_with_any_data']]
bars = ax.barh(
    miss_block['bloque_tematico'],
    miss_block['n_countries_with_any_data'],
    color=colors
)
ax.axvline(x=199, color='navy', linestyle='--', alpha=0.5, label='Total (199)')
ax.axvline(x=28, color='red', linestyle=':', alpha=0.8, label='~28 (IAPP)')
ax.set_xlabel('N países con al menos 1 dato en el bloque')
ax.set_title('Países con Datos por Bloque Temático')
ax.legend()
for bar, val in zip(bars, miss_block['n_countries_with_any_data']):
    ax.text(val + 2, bar.get_y() + bar.get_height()/2,
            f'{val}', va='center', fontsize=10)

# Panel derecho: cobertura promedio (%)
ax2 = axes[1]
colors2 = ['#c0392b' if v < 30 else '#e67e22' if v < 70 else '#27ae60'
           for v in miss_block['pct_complete_mean']]
bars2 = ax2.barh(
    miss_block['bloque_tematico'],
    miss_block['pct_complete_mean'],
    color=colors2
)
ax2.axvline(x=70, color='green', linestyle='--', alpha=0.5, label='70% (umbral ok)')
ax2.axvline(x=30, color='red', linestyle=':', alpha=0.8, label='30% (umbral mínimo)')
ax2.set_xlabel('Cobertura promedio por variable (%)')
ax2.set_title('Cobertura Promedio de Variables por Bloque')
ax2.legend()
for bar, val in zip(bars2, miss_block['pct_complete_mean']):
    ax2.text(val + 0.5, bar.get_y() + bar.get_height()/2,
             f'{val:.1f}%', va='center', fontsize=10)

plt.suptitle('Diagnóstico de Cobertura por Bloque Temático\nMatriz Madre Fase 3 v1.1 — 199 países',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.figtext(0.5, -0.02,
            'Fuente: Matriz Madre Fase 3 v1.1. ROJO: <50 países o <30% cobertura. '
            'NARANJA: zona de advertencia. VERDE: cobertura adecuada.',
            ha='center', fontsize=9, color='gray')
plt.show()

print()
print("MENSAJE CLAVE:")
print("  regulatory_treatment (IAPP) tiene datos para ~28/199 países (~14%).")
print("  Este es el cuello de botella estructural de la investigación.")
print("  tech_infrastructure_control tiene cobertura casi universal (~100%).")

In [ ]:
# Heatmap: cobertura por bloque y región geográfica
miss_country = coverage_results['eda_missingness_by_country']

# Construir tabla: region × bloque con cobertura promedio
region_block_rows = []
for region, grp in miss_country.groupby('region', dropna=False):
    row = {'region': str(region)}
    for block in BLOCKS:
        col = f'pct_{block}'
        if col in grp.columns:
            row[block] = grp[col].mean()
        else:
            row[block] = np.nan
    region_block_rows.append(row)

heatmap_df = pd.DataFrame(region_block_rows).set_index('region')

fig, ax = plt.subplots(figsize=(14, 7))
sns.heatmap(
    heatmap_df,
    annot=True, fmt='.1f',
    cmap='RdYlGn',
    vmin=0, vmax=100,
    linewidths=0.5,
    ax=ax,
    cbar_kws={'label': 'Cobertura promedio (%)'}
)
ax.set_title('Cobertura por Bloque Temático y Región Geográfica (%)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Bloque Temático')
ax.set_ylabel('Región Geográfica (Banco Mundial)')
plt.xticks(rotation=30, ha='right')
plt.yticks(rotation=0)
plt.figtext(0.5, -0.04,
            'Fuente: Matriz Madre Fase 3 v1.1. Verde=cobertura alta, Rojo=cobertura baja. '
            'Valores en porcentaje (0-100).',
            ha='center', fontsize=9, color='gray')
plt.tight_layout()
plt.show()

In [ ]:
# Top 20 países con mejor cobertura global
top_20_coverage = miss_country[['iso3', 'country_name_canonical', 'region',
                                 'income_group', 'pct_vars_available',
                                 'n_vars_available', 'n_blocks_with_data']].head(20)
print("Top 20 países por cobertura de variables (%):")
display(top_20_coverage)


In [ ]:
# Cobertura de Chile por bloque
chile_coverage = miss_country[miss_country['iso3'] == 'CHL']
block_pct_cols = [f'pct_{b}' for b in BLOCKS if f'pct_{b}' in miss_country.columns]
chile_cov_blocks = chile_coverage[['iso3', 'country_name_canonical', 'pct_vars_available'] + block_pct_cols]
print("Cobertura de Chile — global y por bloque temático:")
display(chile_cov_blocks)

chile_vals = [chile_coverage[f'pct_{b}'].values[0] if f'pct_{b}' in chile_coverage.columns else 0 for b in BLOCKS]
fig, ax = plt.subplots(figsize=(10, 5))
bar_colors = ['#c0392b' if v < 30 else '#e67e22' if v < 70 else '#2980b9' for v in chile_vals]
bars = ax.bar(BLOCKS, chile_vals, color=bar_colors)
ax.axhline(y=70, color='green', linestyle='--', alpha=0.6, label='70% umbral')
ax.set_ylim(0, 110)
ax.set_ylabel('Cobertura de variables (%)')
ax.set_title('Chile — Cobertura por Bloque Temático', fontweight='bold')
ax.legend()
plt.xticks(rotation=25, ha='right')
for bar, val in zip(bars, chile_vals):
    ax.text(bar.get_x() + bar.get_width()/2, val + 1, f'{val:.1f}%', ha='center', fontsize=10)
plt.tight_layout()
plt.show()


---
## Bloque C — Estadística Univariada Robusta

### ¿Por qué estadística robusta?

Las variables de ecosistema IA tienen **distribuciones muy asimétricas**. Por ejemplo:
- La inversión privada en IA de USA es 10-100× mayor que la de cualquier otro país.
- El número de patentes de IA de China es un outlier estructural.

En estas condiciones:
- La **media** es dominada por outliers y no representa al país típico.
- La **desviación estándar** infla artificialmente la variabilidad percibida.
- Los tests de normalidad (Shapiro-Wilk) fallarán casi siempre.

Por eso usamos estadísticos **robustos**:
- **Mediana** en lugar de media — no le da más peso a los extremos.
- **MAD** (Desviación Mediana Absoluta) en lugar de std — robusto a outliers.
- **IQR** para definir outliers — basado en percentiles, no en distancia a la media.
- **Z-score robusto** (basado en MAD) para detectar outliers extremos.

### Candidatos para transformación logarítmica

Variables con asimetría |skewness| > 2.0 son candidatas a transformación `log(1+x)`.
Esto no es imputación — es una transformación de escala que facilita la estimación lineal.

In [ ]:
# Ejecutar análisis univariado
print("Ejecutando análisis univariado (puede tardar ~1-2 min)...")
univariate_results = run_univariate_analysis(wide=wide, dictionary=dictionary, save=False)
print("✓ Análisis univariado completado")
print(f"  Variables analizadas: {len(univariate_results['eda_variable_summary'])}")
print(f"  Outliers detectados: {len(univariate_results['eda_outliers'])} instancias")


In [ ]:
# Top 20 variables con mayor completitud
var_summary = univariate_results['eda_variable_summary']
var_ok = var_summary[var_summary['status'] == 'ok'].copy()
top_complete = var_ok.nlargest(20, 'pct_complete')[
    ['variable_matriz', 'bloque_tematico', 'source_id', 'pct_complete', 'n', 'median', 'mad', 'skewness', 'suggest_log_transform']
]
print("Top 20 variables con mayor completitud:")
display(top_complete)


In [ ]:
# Distribución de asimetría (skewness) entre todas las variables
skew_vals = var_ok['skewness'].dropna()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma
ax = axes[0]
ax.hist(skew_vals, bins=40, color='steelblue', edgecolor='white', alpha=0.85)
ax.axvline(x=2.0, color='red', linestyle='--', alpha=0.7, label='|skewness|=2 (umbral)')
ax.axvline(x=-2.0, color='red', linestyle='--', alpha=0.7)
ax.axvline(x=0, color='green', linestyle='-', alpha=0.5, label='simetría')
ax.set_xlabel('Skewness (asimetría)')
ax.set_ylabel('N variables')
ax.set_title('Distribución de Asimetría\nentre todas las variables analíticas')
ax.legend()

n_log = (skew_vals.abs() > 2.0).sum()
ax.text(0.97, 0.97, f'{n_log} variables con |skew|>2\n(candidatas a log-transform)',
        transform=ax.transAxes, ha='right', va='top',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8), fontsize=9)

# Boxplot por bloque
ax2 = axes[1]
block_skew_data = []
block_labels = []
for block in BLOCKS:
    block_df = var_ok[var_ok['bloque_tematico'] == block]['skewness'].dropna()
    if len(block_df) > 0:
        block_skew_data.append(block_df.values)
        block_labels.append(block.replace('_', '\n'))

if block_skew_data:
    ax2.boxplot(block_skew_data, labels=block_labels, vert=True,
                patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.6))
    ax2.axhline(y=2.0, color='red', linestyle='--', alpha=0.7, label='|skew|=2')
    ax2.axhline(y=-2.0, color='red', linestyle='--', alpha=0.7)
    ax2.set_ylabel('Skewness')
    ax2.set_title('Asimetría por Bloque Temático')
    ax2.legend()

plt.suptitle('Diagnóstico de Asimetría — Variables Numéricas',
             fontsize=13, fontweight='bold', y=1.01)
plt.figtext(0.5, -0.03,
            'Fuente: Matriz Madre Fase 3 v1.1. Variables con |skewness|>2 son candidatas '
            'a transformación log(1+x) antes del modelado en Fase 6.',
            ha='center', fontsize=9, color='gray')
plt.tight_layout()
plt.show()

In [ ]:
# Top 10 variables más asimétricas (candidatas a transformación log)
top_skewed = var_ok.nlargest(10, 'skewness')[
    ['variable_matriz', 'bloque_tematico', 'source_id', 'pct_complete', 'skewness', 'n_outliers_iqr_1_5', 'suggest_log_transform']
]
print("Top 10 variables más asimétricas (candidatas a log-transform en Fase 5):")
display(top_skewed)


In [ ]:
# Outliers para países clave
outliers = univariate_results['eda_outliers']
key_countries = ['CHL', 'SGP', 'ARE', 'IRL', 'USA', 'CHN', 'GBR', 'DEU']

if not outliers.empty:
    outliers_key = outliers[outliers['iso3'].isin(key_countries)].copy()
    outliers_key_top = outliers_key.groupby('iso3').size().reset_index(name='n_outliers')
    outliers_key_top = outliers_key_top.sort_values('n_outliers', ascending=False)

    print("Número de outliers extremos (IQR×3) por país clave:")
    display(outliers_key_top)

    # Detalle de outliers de Chile
    chile_outliers = outliers_key[outliers_key['iso3'] == 'CHL'][
        ['variable_matriz', 'bloque_tematico', 'value',
         'lower_fence', 'upper_fence', 'direction']
    ]
    if len(chile_outliers) > 0:
        print(f"\nOutliers extremos de Chile ({len(chile_outliers)} total):")
        display(chile_outliers)
    else:
        print("\nChile no tiene outliers extremos (IQR×3) — posición típica en distribuciones.")
else:
    print("No se detectaron outliers extremos.")

---
## Bloque D — Análisis Intra-Bloque

### ¿Qué es un bloque temático?

La Matriz Madre organiza las 397 variables analíticas en **6 bloques temáticos**:

| Bloque | Descripción | Rol en el modelo causal |
|--------|-------------|------------------------|
| `regulatory_treatment` | Leyes, políticas y marcos regulatorios de IA | Variable X₁ (tratamiento) |
| `ecosystem_outcome` | Inversión, patentes, adopción IA en el país | Variable Y (resultado) |
| `adoption_diffusion` | Difusión de tecnología IA en empresas/hogares | Variable Y auxiliar / mediador |
| `socioeconomic_control` | PIB, educación, urbanización | Variable X₂ (control) |
| `institutional_control` | Rule of law, efectividad gubernamental | Variable X₂ (control) |
| `tech_infrastructure_control` | Internet, R&D, capital humano tech | Variable X₂ (control) |

Este bloque analiza qué variables dentro de cada bloque tienen la mayor cobertura
y variabilidad — las candidatas naturales para representar al bloque en el modelo.

In [ ]:
# Análisis intra-bloque: top variables por cobertura y variabilidad
var_ok = univariate_results['eda_variable_summary']
if 'status' in var_ok.columns:
    var_ok = var_ok[var_ok['status'] == 'ok'].copy()

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

print("=" * 80)
print("ANÁLISIS INTRA-BLOQUE — Variables más informativas por bloque")
print("=" * 80)

for i, block in enumerate(BLOCKS):
    block_df = var_ok[var_ok['bloque_tematico'] == block].copy()
    n_vars = len(block_df)
    avg_cov = block_df['pct_complete'].mean() if n_vars > 0 else 0

    print(f"\n{'─'*60}")
    print(f"Bloque: {block}")
    print(f"  Variables: {n_vars} | Cobertura promedio: {avg_cov:.1f}%")

    if n_vars > 0:
        top5 = block_df.nlargest(5, 'pct_complete')[
            ['variable_matriz', 'pct_complete', 'source_id', 'skewness']
        ]
        print("  Top 5 variables por cobertura:")
        print(top5.to_string(index=False))

        # Mini gráfico de barras
        ax = axes[i]
        top10 = block_df.nlargest(10, 'pct_complete')
        short_names = [v[:30] + '...' if len(v) > 30 else v
                       for v in top10['variable_matriz']]
        bar_colors = ['#2980b9' if v >= 70 else '#e67e22' if v >= 30 else '#c0392b'
                      for v in top10['pct_complete']]
        ax.barh(range(len(top10)), top10['pct_complete'], color=bar_colors)
        ax.set_yticks(range(len(top10)))
        ax.set_yticklabels(short_names, fontsize=7)
        ax.axvline(x=70, color='green', linestyle='--', alpha=0.5)
        ax.set_xlabel('Cobertura (%)')
        ax.set_title(f'{block}\n({n_vars} vars, avg {avg_cov:.0f}%)',
                     fontsize=9, fontweight='bold')
        ax.set_xlim(0, 105)

plt.suptitle('Top 10 Variables por Cobertura en Cada Bloque Temático',
             fontsize=13, fontweight='bold', y=1.01)
plt.figtext(0.5, -0.02,
            'Fuente: Matriz Madre Fase 3 v1.1. Azul: >70% cobertura (ok). '
            'Naranja: 30-70% (usar con cautela). Rojo: <30% (excluir o revisar).',
            ha='center', fontsize=9, color='gray')
plt.tight_layout()
plt.show()

---
## Bloque D/K — Análisis por Bloque, PCA y Clustering Exploratorio

Este bloque genera los summaries por bloque temático, PCA exploratorio, coordenadas 2D para visualización y clusters regulatorios. Las componentes y clusters son cartografía descriptiva; no se convierten en features finales en Fase 4.


In [ ]:
# Ejecutar análisis por bloque, PCA y clustering exploratorio
print("Ejecutando análisis por bloque/PCA/clustering...")
block_results = run_block_analysis(wide=wide, dictionary=dictionary, save=False)
print("Análisis por bloque completado")
print(f"Tablas generadas: {list(block_results.keys())}")

display(block_results['eda_block_pca_loadings'].head(20))



---
## Bloque E/F — Correlaciones Spearman y Corrección FDR

### ¿Por qué Spearman y no Pearson?

La correlación de **Pearson** asume que ambas variables son aproximadamente normales.
Como vimos en el Bloque C, nuestras variables tienen distribuciones muy asimétricas.
La correlación de **Spearman** trabaja sobre los rangos de los datos — es robusta a
outliers y funciona bien con distribuciones no normales.

### El problema del testing múltiple

Con ~397 variables analíticas, calculamos potencialmente **~78,000 pares de correlaciones**.
Si usamos el criterio p<0.05 sin corrección, esperamos que **~3,900 correlaciones sean
'significativas' por puro azar** (5% de falsos positivos).

La corrección **FDR de Benjamini-Hochberg** controla la tasa de falsos descubrimientos:
garantiza que entre las correlaciones que reportamos como significativas, a lo sumo
el 5% son falsas. Es más potente que la corrección de Bonferroni (que controla el
FWER y es demasiado conservadora con tantas pruebas).

### Redundancia

Pares con |ρ| > 0.85 son candidatos a **colapso** (Feature Engineering en Fase 5):
si dos variables miden casi lo mismo, el modelo solo necesita una.

> **Recordatorio importante:** Toda correlación en este bloque es descriptiva, no causal.
> Una correlación alta entre regulación y ecosistema no implica que la regulación cause el ecosistema.

In [ ]:
# Ejecutar análisis bivariado
# ADVERTENCIA: Este análisis puede tardar 5-15 minutos con muchas variables.
# Los resultados se guardan en outputs/eda_principal/ para no recalcular.
print("Ejecutando análisis bivariado (puede tardar varios minutos)...")
bivariate_results = run_bivariate_analysis(wide=wide, dictionary=dictionary, save=False)
print("✓ Análisis bivariado completado")

spearman_df = bivariate_results['eda_correlations_spearman']
redundancy_df = bivariate_results['eda_redundancy_report']
inter_block_df = bivariate_results['eda_inter_block_correlations']

print(f"\nResumen:")
print(f"  Pares de correlaciones calculados: {len(spearman_df):,}")
if 'survives_fdr_05' in spearman_df.columns:
    n_fdr = spearman_df['survives_fdr_05'].sum()
    pct_fdr = n_fdr / len(spearman_df) * 100 if len(spearman_df) > 0 else 0
    print(f"  Pares que sobreviven FDR (q<0.05): {n_fdr:,} ({pct_fdr:.1f}%)")
print(f"  Pares redundantes (|rho|>=0.85): {len(redundancy_df):,}")
print(f"  Correlaciones inter-bloque calculadas: {len(inter_block_df):,}")


In [ ]:
# Top 20 correlaciones inter-bloque: regulatory × ecosystem (y otros pares clave)
if not inter_block_df.empty:
    top_inter = inter_block_df.head(20)[
        ['block_a', 'block_b', 'var_a', 'var_b', 'rho_spearman', 'pvalue_raw', 'pvalue_fdr', 'survives_fdr_05', 'n_pairs']
    ] if 'rho_spearman' in inter_block_df.columns else inter_block_df.head(20)
    print("Top 20 correlaciones inter-bloque (ordenadas por |rho| descendente):")
    display(top_inter)
else:
    print("No se calcularon correlaciones inter-bloque (cobertura insuficiente).")


In [ ]:
# Resumen visual: distribución de correlaciones y FDR
if not spearman_df.empty and 'rho' in spearman_df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Distribución de rho
    ax = axes[0]
    if 'survives_fdr_05' in spearman_df.columns:
        rho_sig = spearman_df[spearman_df['survives_fdr_05']]['rho']
        rho_not = spearman_df[~spearman_df['survives_fdr_05']]['rho']
        ax.hist(rho_not, bins=50, color='lightgray', alpha=0.7,
                label=f'No significativo ({len(rho_not):,})', density=True)
        ax.hist(rho_sig, bins=50, color='steelblue', alpha=0.8,
                label=f'Sobrevive FDR ({len(rho_sig):,})', density=True)
    else:
        ax.hist(spearman_df['rho'], bins=50, color='steelblue', alpha=0.8)
    ax.axvline(x=0, color='black', linestyle='-', alpha=0.5)
    ax.set_xlabel('Correlación de Spearman (rho)')
    ax.set_ylabel('Densidad')
    ax.set_title('Distribución de Correlaciones Spearman')
    ax.legend(fontsize=9)

    # Heatmap de correlaciones inter-bloque (solo si tenemos datos suficientes)
    ax2 = axes[1]
    if not inter_block_df.empty and 'rho_spearman' in inter_block_df.columns:
        # Pivot: block_a × block_b con rho medio
        pivot_data = inter_block_df.groupby(['block_a', 'block_b'])['rho_spearman'].mean().reset_index()
        pivot_table = pivot_data.pivot(index='block_a', columns='block_b', values='rho_spearman')
        if not pivot_table.empty:
            sns.heatmap(
                pivot_table,
                annot=True, fmt='.2f',
                cmap='RdBu_r', vmin=-1, vmax=1,
                linewidths=0.5,
                ax=ax2,
                cbar_kws={'label': 'rho promedio'}
            )
            ax2.set_title('Correlación Media Inter-Bloque (Spearman)')
            ax2.set_xlabel('')
            ax2.set_ylabel('')
            plt.setp(ax2.get_xticklabels(), rotation=25, ha='right', fontsize=8)
            plt.setp(ax2.get_yticklabels(), rotation=0, fontsize=8)
        else:
            ax2.text(0.5, 0.5, 'Datos insuficientes para heatmap',
                     ha='center', va='center', transform=ax2.transAxes)
    else:
        ax2.text(0.5, 0.5, 'Correlaciones inter-bloque no disponibles',
                 ha='center', va='center', transform=ax2.transAxes)

    plt.suptitle('Análisis de Correlaciones — Spearman + FDR',
                 fontsize=13, fontweight='bold', y=1.01)
    plt.figtext(0.5, -0.03,
                'Fuente: Matriz Madre Fase 3 v1.1. Corrección FDR: Benjamini-Hochberg (q<0.05). '
                'Toda correlación es descriptiva, no causal.',
                ha='center', fontsize=9, color='gray')
    plt.tight_layout()
    plt.show()

In [ ]:
# Reporte de redundancia: pares con |rho| >= 0.85
if not redundancy_df.empty:
    top_redundant = redundancy_df.head(20)[
        [c for c in ['var_a', 'var_b', 'rho', 'bloque_a', 'bloque_b', 'same_block', 'is_primary_a', 'is_primary_b', 'recommendation'] if c in redundancy_df.columns]
    ]
    print(f"Pares redundantes detectados (|rho| >= 0.85): {len(redundancy_df)}")
    print("Top 20 — candidatos a colapso en Feature Engineering (Fase 5):")
    display(top_redundant)
else:
    print("No se detectaron pares redundantes (|rho| >= 0.85).")


---
## Bloque G/H — Sub-preguntas y Taxonomías Exploratorias

Aquí se materializa la conexión entre EDA y las cuatro preguntas del estudio. El mapeo identifica variables observadas, cobertura efectiva y gaps; las taxonomías binding/enabling son insumos auditables para Fase 5, no índices definitivos.


In [ ]:
# Mapear variables reales a Q1-Q4 y diagnosticar viabilidad
print("Mapeando sub-preguntas Q1-Q4...")
question_results = run_question_mapping(wide=wide, dictionary=dictionary, save=False)
display(question_results['eda_question_viability'])

print("Construyendo taxonomías binding/enabling...")
taxonomy_results = run_taxonomy_analysis(wide=wide, dictionary=dictionary, save=False)
display(taxonomy_results['eda_binding_classification'].head(10))
print("Mapeo y taxonomías completados")



---
## Bloque I — Perfiles de País

### Metodología de perfilamiento

Para cada país calculamos un **Z-score robusto** (basado en mediana y MAD) por bloque:
se promedia el Z-score de todas las variables numéricas de ese bloque.

Un Z-score positivo significa que el país está **por encima de la mediana mundial** en ese bloque;
negativo significa que está por debajo. Un Z-score de +1.5 en `ecosystem_outcome` significa que
el ecosistema de IA del país es ~1.5 desviaciones medianas absolutas por encima del típico.

### Chile como caso focal

Chile es el **caso focal** de esta investigación (n=1 en el análisis de política pública).
Los perfiles descriptivos de Chile no son datos estadísticos — son el contexto para
entender dónde está Chile en el mundo cuando se debate el Boletín 16821-19.

Comparamos Chile con:
1. **LATAM peers:** ARG, BRA, COL, MEX, PER, URY — contexto regional
2. **Líderes de IA:** SGP (Singapur), ARE (Emiratos), IRL (Irlanda), EST (Estonia) — aspiracionales
3. **Grandes potencias:** USA, GBR, KOR — referencia global

In [ ]:
# Ejecutar análisis de países
print("Ejecutando análisis de perfiles de país...")
country_results = run_country_analysis(
    wide=wide,
    dictionary=dictionary,
    peer_iso3=['ARG', 'BRA', 'COL', 'MEX', 'PER', 'URY', 'SGP', 'ARE', 'IRL', 'EST', 'USA', 'GBR', 'KOR'],
    save=False
)
print("✓ Análisis de países completado")


In [ ]:
# Ranking global: top 20 países por Z-score de ecosystem_outcome
profiles = country_results['eda_country_profiles']
eco_col = 'zscore_ecosystem_outcome_mean'
rank_col = 'rank_ecosystem_outcome'
display_cols = ['iso3', 'country_name_canonical', 'region', 'income_group']
for col in [eco_col, rank_col, 'zscore_regulatory_treatment_mean', 'zscore_socioeconomic_control_mean', 'zscore_tech_infrastructure_control_mean']:
    if col in profiles.columns:
        display_cols.append(col)
top_20_eco = profiles[display_cols].head(20)
print("Top 20 países por Z-score de ecosistema de IA (ecosystem_outcome):")
display(top_20_eco)


In [ ]:
# Perfil detallado de Chile
chile_profile_dict = country_results.get('chile_profile_dict', {})

if chile_profile_dict:
    print("Perfil de Chile — Z-scores y rankings por bloque:")
    profile_rows = []
    for block in BLOCKS:
        row = {'bloque': block}
        for key in [f'zscore_{block}', f'avg_percentile_{block}',
                    f'pct_coverage_{block}', f'n_data_{block}']:
            if key in chile_profile_dict:
                row[key.replace(f'_{block}', '')] = chile_profile_dict[key]
        n_countries = 199
        rank_key = f'rank_{block}_of_{n_countries}'
        if rank_key in chile_profile_dict:
            row['ranking_mundial'] = f"#{chile_profile_dict[rank_key]}/199"
        profile_rows.append(row)

    chile_profile_df = pd.DataFrame(profile_rows)
    display(chile_profile_df)
else:
    print("Perfil de Chile no disponible — verificar datos.")

# Tabla adicional desde country_profiles
chile_in_profiles = profiles[profiles['iso3'] == 'CHL']
if not chile_in_profiles.empty:
    zscore_cols = [c for c in chile_in_profiles.columns if c.startswith('zscore_')]
    rank_cols_list = [c for c in chile_in_profiles.columns if c.startswith('rank_')]
    chile_summary = chile_in_profiles[['iso3', 'region', 'income_group'] + zscore_cols].copy()
    print("\nZ-scores de Chile por bloque (valores numéricos):")
    display(chile_summary)

In [ ]:
# Chile vs peers LATAM
chile_vs_peers = country_results['eda_chile_vs_peers']
latam_peers = ['ARG', 'BRA', 'CHL', 'COL', 'MEX', 'PER', 'URY']
latam_df = chile_vs_peers[chile_vs_peers['iso3'].isin(latam_peers)].copy()
if not latam_df.empty:
    zscore_block_cols = [c for c in latam_df.columns if c.startswith('zscore_mean_')]
    display_cols_latam = ['iso3', 'country_name', 'income_group'] + zscore_block_cols
    display_cols_latam = [c for c in display_cols_latam if c in latam_df.columns]
    print("Chile vs peers LATAM — Z-scores por bloque (relativo al universo de 199 países):")
    display(latam_df[display_cols_latam])


In [ ]:
# Deep dive: SGP, ARE, IRL vs Chile
deep_dives = country_results['eda_singapore_uae_ireland_profiles']
if not deep_dives.empty:
    print("Perfiles expandidos — Chile vs líderes de IA (SGP, ARE, IRL):")
    cov_cols = [c for c in deep_dives.columns if c.startswith('pct_coverage_')]
    rank_cols_dd = [c for c in deep_dives.columns if c.startswith('avg_pct_rank_')]
    display_cols_dd = ['iso3', 'country_name', 'region', 'income_group'] + cov_cols + rank_cols_dd
    display_cols_dd = [c for c in display_cols_dd if c in deep_dives.columns]
    display(deep_dives[display_cols_dd])


In [ ]:
# Gráfico de barras: Chile vs países clave por bloque
compare_iso3 = ['CHL', 'SGP', 'ARE', 'IRL', 'ARG', 'BRA', 'MEX']
compare_df = profiles[profiles['iso3'].isin(compare_iso3)].copy()

zscore_cols_plot = [c for c in profiles.columns
                    if c.startswith('zscore_') and c.endswith('_mean')]

if not compare_df.empty and zscore_cols_plot:
    fig, ax = plt.subplots(figsize=(14, 7))

    # Reorganizar para plotting
    compare_pivot = compare_df.set_index('iso3')[zscore_cols_plot]
    # Acortar nombres de columnas
    short_cols = [c.replace('zscore_', '').replace('_mean', '').replace('_', '\n')
                  for c in zscore_cols_plot]
    compare_pivot.columns = short_cols

    x = np.arange(len(short_cols))
    n_countries = len(compare_pivot)
    width = 0.8 / n_countries
    country_colors = {
        'CHL': '#e74c3c',  # Chile destacado en rojo
        'SGP': '#2ecc71',
        'ARE': '#3498db',
        'IRL': '#9b59b6',
        'ARG': '#f39c12',
        'BRA': '#1abc9c',
        'MEX': '#e67e22',
    }

    for i, (iso3_code, row) in enumerate(compare_pivot.iterrows()):
        offset = (i - n_countries / 2 + 0.5) * width
        color = country_colors.get(iso3_code, '#95a5a6')
        linewidth = 2.5 if iso3_code == 'CHL' else 1
        alpha = 1.0 if iso3_code == 'CHL' else 0.75
        ax.bar(x + offset, row.values, width * 0.9,
               label=iso3_code, color=color, alpha=alpha,
               edgecolor='black' if iso3_code == 'CHL' else 'none',
               linewidth=linewidth)

    ax.axhline(y=0, color='black', linestyle='-', alpha=0.3, linewidth=1)
    ax.set_xticks(x)
    ax.set_xticklabels(short_cols, fontsize=9)
    ax.set_ylabel('Z-score robusto (mediana=0, MAD=1)')
    ax.set_title('Chile vs Países Clave — Z-scores por Bloque Temático\n'
                 'Valores positivos = por encima de la mediana mundial',
                 fontweight='bold')
    ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=10)
    plt.figtext(0.5, -0.03,
                'Fuente: Matriz Madre Fase 3 v1.1. Chile destacado en rojo con borde. '
                'Z-score robusto: (valor - mediana) / MAD calculado sobre 199 países.',
                ha='center', fontsize=9, color='gray')
    plt.tight_layout()
    plt.show()
else:
    print("Datos insuficientes para el gráfico comparativo de países.")

---
## Bloque J — Sensibilidad Temporal

El snapshot de Fase 3 resume un panel 2018-2026. Este bloque revisa años usados, estabilidad temporal y diferencias snapshot vs año previo sin modelar el panel completo.


In [ ]:
# Ejecutar sensibilidad temporal snapshot vs panel
print("Ejecutando sensibilidad temporal...")
temporal_results = run_temporal_analysis(save=False)
print("Sensibilidad temporal completada")
display(temporal_results['eda_year_used_distribution'].head(20))



---
## Bloque L — Submuestras Candidatas (Multiverse Analysis)

### ¿Por qué NOT elegir una sola muestra ahora?

Esta es quizás la decisión metodológica más importante del proyecto.

En ciencias sociales, la elección de muestra es una **decisión del analista** que puede
influir fuertemente en los resultados. Si elegimos la muestra *después* de ver los resultados,
corremos el riesgo de **specification searching** o **p-hacking**: encontrar la muestra
que confirma nuestra hipótesis favorita.

La solución es el **multiverse analysis** (Simonsohn et al. 2020; Steegen et al. 2016):
1. **Preregistrar** múltiples submuestras candidatas con criterios explícitos.
2. **Modelar** con la muestra principal en Fase 6.
3. **Verificar robustez** modelando con todas las submuestras en Fase 7.
4. Si un resultado **solo aparece con una submuestra**, es evidencia débil.

### Las 6 submuestras candidatas

| Submuestra | Criterio | Fortaleza | Limitación |
|------------|----------|-----------|------------|
| `regulada` | ≥1 variable IAPP no nula | Tratamiento directo | N~28 (baja potencia) |
| `densa_80` | ≥80% cobertura de variables | Alta calidad datos | N reducido |
| `densa_60` | ≥60% cobertura de variables | Balance calidad/N | Algo de missingness |
| `comparable_chile` | Top-K países más similares a Chile (Gower) | Validez interna | Riesgo sesgo selección |
| `oecd_plus_latam` | OECD + LATAM (lista cerrada) | Comparabilidad institucional | Sesgo hacia desarrollados |
| `full` | Todos los 199 países soberanos | Máxima potencia | Alta heterogeneidad |

> **REGLA DE ORO:** "La elección de muestra se decide en Fase 6. Cualquier resultado que
> 'solo funciona' con una submuestra debe ser cuestionado."

**Referencias:**
- Simonsohn, Simmons & Nelson (2020). Specification Curve Analysis. *Nature Human Behaviour.*
- Steegen, Tuerlinckx, Gelman & Vanpaemel (2016). Increasing Transparency Through a Multiverse Analysis. *Perspectives on Psychological Science.*

In [ ]:
# Ejecutar análisis de submuestras
print("Construyendo 6 submuestras candidatas...")
submuestras_results = run_submuestras_analysis(wide=wide, dictionary=dictionary, save=False)
print("✓ Submuestras calculadas")

submuestras_summary = submuestras_results['eda_submuestras_candidatas']
membership_df = submuestras_results['eda_submuestra_membership']


In [ ]:
# Tabla resumen de submuestras
display_cols_sm = ['submuestra', 'n_countries', 'chile_present', 'key_countries_present', 'key_countries_total', 'pct_coverage_mean', 'criterion']
display_cols_sm = [c for c in display_cols_sm if c in submuestras_summary.columns]
print("Resumen de 6 submuestras candidatas (multiverse analysis preregistrado):")
display(submuestras_summary[display_cols_sm])


In [ ]:
# Gráfico: n_países por submuestra con cobertura
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Panel izquierdo: N países
ax = axes[0]
sm_names = submuestras_summary['submuestra'].tolist()
sm_n = submuestras_summary['n_countries'].tolist()
colors_sm = ['#e74c3c' if 'regulada' in s else '#3498db'
             if 'densa' in s else '#2ecc71'
             for s in sm_names]
bars = ax.bar(sm_names, sm_n, color=colors_sm, edgecolor='white')
for bar, val in zip(bars, sm_n):
    ax.text(bar.get_x() + bar.get_width()/2, val + 1,
            str(val), ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.set_ylabel('N países en la submuestra')
ax.set_title('Tamaño de Cada Submuestra', fontweight='bold')
ax.axhline(y=30, color='red', linestyle='--', alpha=0.5, label='N mínimo potencia')
ax.legend()
plt.setp(ax.get_xticklabels(), rotation=25, ha='right')

# Panel derecho: cobertura media por submuestra
ax2 = axes[1]
sm_cov = submuestras_summary['pct_coverage_mean'].tolist()
bars2 = ax2.bar(sm_names, sm_cov, color=colors_sm, edgecolor='white')
for bar, val in zip(bars2, sm_cov):
    ax2.text(bar.get_x() + bar.get_width()/2, val + 0.5,
             f'{val:.1f}%', ha='center', va='bottom', fontsize=10)
ax2.set_ylabel('Cobertura media de variables (%)')
ax2.set_title('Cobertura Media por Submuestra', fontweight='bold')
ax2.axhline(y=70, color='green', linestyle='--', alpha=0.5, label='70% umbral')
ax2.set_ylim(0, 110)
ax2.legend()
plt.setp(ax2.get_xticklabels(), rotation=25, ha='right')

plt.suptitle('Submuestras Candidatas — Multiverse Analysis\n'
             'ROJO: regulada (~28 países, tratamiento directo) | '
             'AZUL: densa | VERDE: geográfica/comparativa',
             fontsize=12, fontweight='bold', y=1.02)
plt.figtext(0.5, -0.04,
            'Fuente: Matriz Madre Fase 3 v1.1. Ref: Simonsohn et al. (2020), Steegen et al. (2016). '
            'La selección de la muestra principal se decide en Fase 6.',
            ha='center', fontsize=9, color='gray')
plt.tight_layout()
plt.show()

In [ ]:
# Países clave: ¿en qué submuestras aparecen?
key_iso3 = ['CHL', 'USA', 'SGP', 'ARE', 'IRL', 'EST', 'KOR', 'GBR', 'ARG', 'BRA', 'MEX', 'CHN', 'DEU']
sm_cols = [c for c in submuestras_results['eda_submuestra_membership'].columns if c in ['densa_80', 'densa_60', 'regulada', 'comparable_chile', 'oecd_plus_latam', 'full']]
key_membership = membership_df[membership_df['iso3'].isin(key_iso3)][['iso3', 'country_name_canonical'] + sm_cols]
if not key_membership.empty:
    print("Pertenencia de países clave a cada submuestra (1=incluido, 0=excluido):")
    display(key_membership)


---
## Bloque M — Recomendaciones para Fase 5

### ¿Qué habilita este EDA?

Este análisis exploratorio prepara el terreno para las fases siguientes:

**Fase 5 — Feature Engineering:**
- Variables identificadas para transformación log (asimetría > 2)
- Variables redundantes (|ρ| > 0.85) candidatas a colapso
- Variables de baja cobertura (<30%) candidatas a exclusión

**Fase 6 — Modelado Causal:**
- Especificación Y (outcome): `ecosystem_outcome` + `adoption_diffusion`
- Especificación X₁ (tratamiento): `regulatory_treatment` (IAPP)
- Especificación X₂ (controles): `socioeconomic_control` + `institutional_control` + `tech_infrastructure_control`
- Muestra principal: `regulada` (~28 países)
- Análisis de sensibilidad: las otras 5 submuestras

**Fase 7 — Robustez:**
- Multiverse analysis sobre las 6 submuestras
- Specification curve analysis

### Preguntas que este EDA puede responder
- ¿Dónde está Chile en el mundo en términos de ecosistema de IA? (Bloque I)
- ¿Qué variables tienen suficiente cobertura para ser usadas en modelos? (Bloque B)
- ¿Hay correlaciones bivariadas entre regulación y ecosistema? (Bloque E/F)

### Preguntas que este EDA NO puede responder
- ¿La regulación de IA causa el ecosistema de IA? (requiere modelado causal en Fase 6)
- ¿Qué tipo de regulación es mejor para Chile? (requiere análisis de contenido en Fase 5 NLP)
- ¿Cuál será el efecto del Boletín 16821-19? (requiere análisis de política contrafactual)

In [ ]:
# Ejecutar módulo de reporte
# Este módulo lee los CSVs guardados por los bloques anteriores
# y genera: HTML report, candidates for FE, data gaps, decisions YAML
print("Ejecutando bloque de recomendaciones y reporte...")
reporting_results = run_reporting(save=False)
print("✓ Reporte completado")
print(f"  Artefactos: {list(reporting_results.keys())}")


In [ ]:
# Tabla de gaps de datos — qué falta para responder las preguntas
data_gaps = reporting_results['data_gaps']
print("Gaps de datos identificados — datos faltantes para responder las preguntas de investigación:")
display(data_gaps)


In [ ]:
# Candidatos para Feature Engineering
fe_candidates = reporting_results.get('candidates', pd.DataFrame())

if not fe_candidates.empty and 'fe_recommendation' in fe_candidates.columns:
    fe_summary = fe_candidates['fe_recommendation'].value_counts().reset_index()
    fe_summary.columns = ['recomendacion', 'n_variables']

    print("Resumen de recomendaciones para Feature Engineering (Fase 5):")
    display(fe_summary)

    # Gráfico
    fig, ax = plt.subplots(figsize=(10, 5))
    color_map = {
        'keep_primary': '#27ae60',
        'keep_with_caution': '#f39c12',
        'transform_log': '#3498db',
        'collapse_redundant': '#9b59b6',
        'exclude_low_coverage': '#e74c3c',
        'exclude_constant': '#95a5a6',
    }
    fe_colors = [color_map.get(r, '#bdc3c7') for r in fe_summary['recomendacion']]
    ax.barh(fe_summary['recomendacion'], fe_summary['n_variables'], color=fe_colors)
    ax.set_xlabel('N variables')
    ax.set_title('Recomendaciones de Feature Engineering por Variable',
                 fontweight='bold')
    for i, val in enumerate(fe_summary['n_variables']):
        ax.text(val + 0.5, i, str(val), va='center', fontsize=10)
    plt.figtext(0.5, -0.03,
                'Fuente: análisis univariado + redundancia Fase 4. '
                'Estas recomendaciones se implementan en Fase 5 (Feature Engineering).',
                ha='center', fontsize=9, color='gray')
    plt.tight_layout()
    plt.show()

    # Top candidatos por categoría
    print("\nTop candidatos 'keep_primary' (mantener como variables principales):")
    keep_primary = fe_candidates[fe_candidates['fe_recommendation'] == 'keep_primary'][
        ['variable_matriz', 'bloque_tematico', 'pct_complete', 'source_id']
    ].head(15)
    display(keep_primary)
else:
    print("No se generaron recomendaciones de Feature Engineering.")
    print("Verificar que los análisis anteriores (Bloques C y E/F) se ejecutaron correctamente.")


---
## Conclusiones del EDA Principal

### Lo que aprendimos

**Sobre los datos:**
- La Matriz Madre v1.1 tiene buena cobertura global (~73% promedio) gracias al bloque de infraestructura (casi 100%).
- El cuello de botella estructural es `regulatory_treatment` (IAPP): ~28 países (~14%). Esto define el tamaño máximo de muestra para el análisis causal principal.
- Chile tiene 97.73% de cobertura global — uno de los países mejor cubiertos en LATAM.
- Muchas variables económicas de IA tienen distribuciones muy asimétricas (inversión, patentes) — usar log(1+x) antes del modelado.

**Sobre Chile:**
- Chile está en una posición moderada a nivel mundial — mejor que el promedio LATAM, pero por debajo de los líderes regionales (SGP, ARE, IRL).
- El ecosistema de IA de Chile es comparable al de URY y COL en LATAM.
- Chile carece de datos IAPP detallados sobre su marco regulatorio — el Boletín 16821-19 aún no está vigente.

**Sobre las correlaciones:**
- Se detectaron correlaciones inter-bloque entre controles socioeconómicos y ecosistema de IA — esperado y consistente con la literatura.
- Las correlaciones entre `regulatory_treatment` y `ecosystem_outcome` tienen N pequeño (~28 países) — interpretación con cautela.

### Preguntas abiertas para Fase 5+6

1. **¿La asociación regulación-ecosistema persiste al controlar por GDP per cápita?** (requiere regresión)
2. **¿Qué tipo de regulación (obligatoria vs. principista) se asocia con mayor ecosistema?** (requiere NLP del corpus legal)
3. **¿El efecto varía según el nivel de desarrollo del país?** (requiere análisis de interacción)
4. **¿Los resultados son robustos a la elección de muestra?** (requiere multiverse analysis en Fase 7)

---

**Nota metodológica final:** Este EDA fue diseñado para ser 100% reproducible. Todos los outputs se
guardan en `FASE4/outputs/eda_principal/`. El reporte HTML interactivo se genera en `EDA_Principal_Fase4.html`.
Ningún resultado de este notebook debe interpretarse como evidencia causal.

**Siguientes pasos:** Fase 5 — Feature Engineering y selección de variables para el modelo causal.

In [ ]:
# Resumen final de outputs generados
print("=" * 70)
print("OUTPUTS GENERADOS EN ESTE EDA")
print("=" * 70)

if OUTPUTS_DIR.exists():
    outputs = sorted(OUTPUTS_DIR.iterdir())
    total_size = 0
    print(f"\nDirectorio: {OUTPUTS_DIR}\n")
    for p in outputs:
        size_kb = p.stat().st_size / 1024
        total_size += p.stat().st_size
        print(f"  {p.name:<55} {size_kb:>8.1f} KB")
    print(f"\n  Total: {len(outputs)} archivos, {total_size/1024/1024:.2f} MB")
else:
    print(f"Directorio de outputs no encontrado: {OUTPUTS_DIR}")

print()
print("✓ EDA Principal Fase 4 completado exitosamente")
print(f"  Versión datos: {current_version()}")
print(f"  Países analizados: 199")
print(f"  Columnas en Matriz Madre: 1,203")


---
## Validación Reproducible

La validación final se ejecuta fuera de la lógica del notebook para confirmar que los artefactos generados cumplen el contrato público de Fase 4.


In [ ]:
# Validación final desde el CLI de Fase 4
import subprocess, sys
result = subprocess.run([sys.executable, '-m', 'src.fase4', 'validate'], cwd=str(FASE4_ROOT), text=True, capture_output=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
assert result.returncode == 0, 'La validación CLI de Fase 4 falló'
